In [2]:
import torch

In [3]:
torch.cuda.is_available()

True

In [4]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])

print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [5]:
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")

print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='cuda:0')


In [6]:
tensor_1 = tensor_1.to("cpu")
tensor_2 = tensor_2.to("cuda")
print(tensor_1)
print(tensor_2)
tensor_1 + tensor_2

tensor([1., 2., 3.])
tensor([4., 5., 6.], device='cuda:0')


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [7]:
torch.cuda.device_count()

1

In [8]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, num_outputs)
        )

    def forward(self, x):
        return self.layers(x)

In [9]:
model = NeuralNetwork(num_inputs=2, num_outputs=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [10]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class ToyDataset(Dataset):
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
    def __init__(self, X, y):
        self.labels = y
        self.features = X
    def __len__(self):
        return len(self.features)    

X_train = torch.tensor(
    [
        [-1.2, 3.1],
        [-0.9, 2.9],
        [-0.5, 2.6],
        [2.3, -1.1],
        [2.7, -1.5]
    ]
)
y_train = torch.tensor([0, 0, 0, 1, 1])


train_dataset = ToyDataset(X_train, y_train)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, drop_last=True)

In [11]:
from torch.nn import functional as F

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features = features.to(device)
        labels = labels.to(device)

        logits = model(features)
        
        property = torch.argmax(logits, dim=1)
        print(property)

        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
        f"Batch {batch_idx:03d}/{len(train_loader):03d} | Train/Val Loss: {loss:.2f}")

tensor([0, 0], device='cuda:0')
Epoch: 001/003Batch 000/002 | Train/Val Loss: 0.24
tensor([0, 0], device='cuda:0')
Epoch: 001/003Batch 001/002 | Train/Val Loss: 0.46
tensor([1, 0], device='cuda:0')
Epoch: 002/003Batch 000/002 | Train/Val Loss: 0.00
tensor([1, 0], device='cuda:0')
Epoch: 002/003Batch 001/002 | Train/Val Loss: 0.00
tensor([1, 1], device='cuda:0')
Epoch: 003/003Batch 000/002 | Train/Val Loss: 0.00
tensor([0, 0], device='cuda:0')
Epoch: 003/003Batch 001/002 | Train/Val Loss: 0.00
